# Structural Drivers of Human Developement: A Cross-Country Analytical Assessment 
## Notebook 05:  MySQL Data Load

**Goal:** Load the cleaned master dataset from Python into MySQL
to enable SQL-based analysis and Power BI connectivity.

**Input file:** `processed_data/master_dataset.csv`
**Target:** `hdi` table in MySQL

In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

BASE_DIR = Path("..").resolve()
PROCESSED_DATA = BASE_DIR / "processed_data"

# Load master dataset
master = pd.read_csv(PROCESSED_DATA / "master_dataset.csv")
print(f"Shape: {master.shape}")
print(master.head())

Shape: (186, 11)
  country_code               country    hdi  gdp_per_capita  \
0          AFG           Afghanistan  0.462      377.665627   
1          AGO                Angola  0.591     2860.902519   
2          ALB               Albania  0.789     5867.650962   
3          AND               Andorra  0.884    39780.415299   
4          ARE  United Arab Emirates  0.937    41828.555330   

   mean_years_schooling  education_expenditure_pct_gdp  \
0              2.514790                            NaN   
1              5.844292                       2.385359   
2             10.121144                       2.729770   
3             11.613440                       2.647280   
4             12.773750                            NaN   

   health_expenditure_per_capita  government_effectiveness    gii  \
0                      80.651604                 -1.880035  0.665   
1                     103.945061                 -1.026034  0.520   
2                     506.869202                

## Connect to MySQL and Create Database

## How to Connect

Credentials are stored in a `.env` file (not committed to GitHub). Copy `.env.example.(1)` to `.env` and fill in your values:
```ini
DB_HOST=127.0.0.1
DB_PORT=3306
DB_USER=root
DB_PASSWORD=your_password_here
DB_NAME=hdi
```

Then connect using the following code:

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
# Create SQLAlchemy engine using environment variables
engine = create_engine('mysql+mysqlconnector://', connect_args={
    'host':     os.getenv('DB_HOST'),
    'port':     int(os.getenv('DB_PORT')),
    'user':     os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
})

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Connection successful")

Connection successful


## Verify Load

In [5]:
# Verify row count in MySQL 
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM hdi_master"))
    count = result.fetchone()[0]
    print(f"Rows confirmed in MySQL: {count}")

# Preview loaded data z
with engine.connect() as conn:
    preview = pd.read_sql("SELECT * FROM hdi_master LIMIT 5", conn)
    print(preview)

Rows confirmed in MySQL: 186
  country_code               country    hdi  gdp_per_capita  \
0          AFG           Afghanistan  0.462      377.665627   
1          AGO                Angola  0.591     2860.902519   
2          ALB               Albania  0.789     5867.650962   
3          AND               Andorra  0.884    39780.415299   
4          ARE  United Arab Emirates  0.937    41828.555330   

   mean_years_schooling  education_expenditure_pct_gdp  \
0              2.514790                            NaN   
1              5.844292                       2.385359   
2             10.121144                       2.729770   
3             11.613440                       2.647280   
4             12.773750                            NaN   

   health_expenditure_per_capita  government_effectiveness    gii  \
0                      80.651604                 -1.880035  0.665   
1                     103.945061                 -1.026034  0.520   
2                     506.869202    

## Summary

| Step | Result |
|---|---|
| Database | `hdi` created in MySQL 8.0 |
| Table | `hdi_master`: 186 rows, 11 columns |
| Connection | SQLAlchemy + mysql-connector-python |
| Tool | VS Code Database Client |

